In [1]:
# Cell 1 - Settings: paths, versions, GPU
from pathlib import Path
import time
import torch, ultralytics
from ultralytics import YOLO

HDD      = Path("/media/hudaifa/Ventoy/datasets")
DATA5    = HDD / "bridge5_yolo"                    # 5-class bridge dataset (dacl10k + CODEBRIM)
YOLO_DIR = Path.home() / "drone_ws/yolo"
RUNS     = YOLO_DIR / "runs"                       # absolute path -> no runs/detect/runs nesting
MODELS   = YOLO_DIR / "models"
CLASSES5 = ["crack", "spallation", "efflorescence", "exposed_bars", "corrosion_stain"]

# exactly the same recipe as model B -> a fair comparison (only the model changes)
TRAIN = dict(data=str(DATA5 / "data.yaml"), imgsz=640, epochs=100, patience=30, cache="ram",
             flipud=0.5, device=0, project=str(RUNS), exist_ok=True)

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0))
for f in ["data.yaml", "data_test_dacl.yaml", "data_test_cb.yaml"]:
    print(f"  {f:22s}", "OK" if (DATA5 / f).exists() else "MISSING - is the HDD plugged in?")
print("  model B (current):", "OK" if (MODELS / "bridge5_y11s.pt").exists() else "MISSING")
print("  model A (19 cls): ", "OK" if (MODELS / "dacl19_y11s.pt").exists() else "MISSING")
YOLO("yolo26s.pt")                                 # downloads the newest COCO weights once
print("YOLO26 available")

ultralytics 8.4.174 | torch 2.14.1+cu130 | GPU: NVIDIA GeForce RTX 4060
  data.yaml              OK
  data_test_dacl.yaml    OK
  data_test_cb.yaml      OK
  model B (current): OK
  model A (19 cls):  OK
YOLO26 available


In [2]:
# Cell 2 - Test function + baseline (the current model B)
import pandas as pd

CSV = RUNS / "compare" / "upgrade_comparison.csv"
CSV.parent.mkdir(parents=True, exist_ok=True)
RESULTS = pd.read_csv(CSV).to_dict("records") if CSV.exists() else []   # survives kernel restarts


def evaluate(weights, label):
    """mAP50 per class on the two unseen test sets (same test as notebook 03, cell 10)."""
    row = {"model": label, "weights": str(weights)}
    for part, name in (("dacl", "dacl10k"), ("cb", "CODEBRIM")):
        m = YOLO(str(weights)).val(data=str(DATA5 / f"data_test_{part}.yaml"), split="test", imgsz=640,
                                   device=0, plots=False, verbose=False,
                                   project=str(RUNS / "compare"), name=f"{label}_{part}", exist_ok=True)
        ap = dict(zip(m.box.ap_class_index.tolist(), m.box.ap50.tolist()))
        for i, c in enumerate(CLASSES5):
            row[f"{name} {c}"] = ap.get(i, float("nan"))
        row[f"{name} mAP50"] = m.box.map50
    row["avg mAP50"] = (row["dacl10k mAP50"] + row["CODEBRIM mAP50"]) / 2
    RESULTS[:] = [r for r in RESULTS if r["model"] != label] + [row]
    pd.DataFrame(RESULTS).to_csv(CSV, index=False)
    return row


def show():
    t = pd.DataFrame(RESULTS).set_index("model")
    print("mAP50 on unseen test photos\n")
    print(t[["dacl10k mAP50", "CODEBRIM mAP50", "avg mAP50"]].round(3).to_string())
    print("\nper class (average of both test sets):")
    print(pd.DataFrame({c: (t[f"dacl10k {c}"] + t[f"CODEBRIM {c}"]) / 2 for c in CLASSES5}).round(3).to_string())


evaluate(MODELS / "bridge5_y11s.pt", "B_yolo11s")
show()

Ultralytics 8.4.174 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7796MiB)
YOLO11s summary (fused): 100 layers, 9,414,735 parameters, 0 gradients, 21.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 3.8±8.0 ms, read: 9.5±5.6 MB/s, size: 99.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /media/hudaifa/Ventoy/datasets/bridge5_yolo/labels/test... 864 images, 78 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 864/864 99.2it/s 8.7s<0.1s
val: /media/hudaifa/Ventoy/datasets/bridge5_yolo/images/test/dacl_dacl10k_v2_validation_0555.jpg: 1 duplicate labels removed
val: New cache created: /media/hudaifa/Ventoy/datasets/bridge5_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 9.1it/s 6.0s<0.2s
                   all        864       4563       0.35       0.28      0.241      0

In [3]:
# Cell 3 - Experiment 1: newest architecture, same size -> YOLO26s (COCO weights = transfer learning)
t = time.time()
YOLO("yolo26s.pt").train(name="bridge5_y26s", batch=16, **TRAIN)
print(f"training took {(time.time() - t) / 3600:.1f} h")
evaluate(RUNS / "bridge5_y26s/weights/best.pt", "E1_yolo26s")
show()

New https://pypi.org/project/ultralytics/8.4.179 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7796MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/media/hudaifa/Ventoy/datasets/bridge5_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, m

In [3]:
# Cell 5 - Build an "A11T camera" copy of the dataset (training photos as the real drone camera sees them)
import sys, shutil, random
import cv2, yaml
from tqdm.auto import tqdm
sys.path.insert(0, str(Path.home() / "drone_ws/scripts"))
from analog_cam import AnalogFX

DATA_AN = HDD / "bridge5_analog_yolo"
LOOKS = {"good": 0.3, "typical": 0.5, "bad": 0.2}          # radio-link quality mix
FX = {name: AnalogFX(name, seed=i) for i, name in enumerate(LOOKS)}
SHARE = {"train": 0.5, "val": 0.5, "test": 1.0}             # train/val: half analog, half clean; test: all analog
random.seed(42)

counts = {}
for split, share in SHARE.items():
    (DATA_AN / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATA_AN / "labels" / split).mkdir(parents=True, exist_ok=True)
    n_an = 0
    for src in tqdm(sorted((DATA5 / "images" / split).glob("*.jpg")), desc=split):
        dst = DATA_AN / "images" / split / src.name
        make_analog = random.random() < share
        look = random.choices(list(LOOKS), list(LOOKS.values()))[0]
        if not dst.exists():
            if make_analog:
                cv2.imwrite(str(dst), FX[look](cv2.imread(str(src))), [cv2.IMWRITE_JPEG_QUALITY, 92])
            else:
                shutil.copy(src, dst)
        n_an += make_analog
        lab = DATA5 / "labels" / split / (src.stem + ".txt")
        if lab.exists():
            shutil.copy(lab, DATA_AN / "labels" / split / lab.name)
    counts[split] = n_an

names = yaml.safe_load((DATA5 / "data.yaml").read_text())["names"]
base = dict(path=str(DATA_AN), train="images/train", val="images/val", names=names)
(DATA_AN / "data.yaml").write_text(yaml.safe_dump({**base, "test": "images/test"}))
for part, prefix in (("dacl", "dacl_"), ("cb", "cb_")):          # same two test sets as before
    files = sorted((DATA_AN / "images/test").glob(prefix + "*.jpg"))
    (DATA_AN / f"test_{part}.txt").write_text("\n".join(str(p) for p in files))
    (DATA_AN / f"data_test_{part}.yaml").write_text(yaml.safe_dump({**base, "test": f"test_{part}.txt"}))

print("analog images made:", counts)
print("dataset ready:", DATA_AN)
ex = sorted((DATA5 / "images/test").glob("dacl_*.jpg"))[0]               # one before/after example
side = cv2.hconcat([cv2.resize(cv2.imread(str(ex)), (480, 360)),
                    cv2.resize(cv2.imread(str(DATA_AN / "images/test" / ex.name)), (480, 360))])
cv2.imwrite(str(DATA_AN / "example_clean_vs_A11T.jpg"), side)
print("example:", DATA_AN / "example_clean_vs_A11T.jpg")

/home/hudaifa/drone_ws/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
test: 100%|████████████████████████████████| 1033/1033 [00:11<00:00, 91.97it/s]


analog images made: {'train': 3216, 'val': 404, 'test': 1033}
dataset ready: /media/hudaifa/Ventoy/datasets/bridge5_analog_yolo
example: /media/hudaifa/Ventoy/datasets/bridge5_analog_yolo/example_clean_vs_A11T.jpg


In [4]:
# Cell 6 - How do the current models do on A11T-look test photos? (baseline for the analog training)
def evaluate_A11T(weights, label):
    global DATA5
    saved, DATA5 = DATA5, DATA_AN
    try:
        return evaluate(weights, label + "_A11T")
    finally:
        DATA5 = saved

evaluate_A11T(MODELS / "bridge5_y11s.pt", "B_yolo11s")
evaluate_A11T(MODELS / "bridge5_y26s.pt", "E1_yolo26s")
show()

Ultralytics 8.4.174 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7796MiB)
YOLO11s summary (fused): 100 layers, 9,414,735 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7044.1±1579.7 MB/s, size: 79.6 KB)
val: Scanning /media/hudaifa/Ventoy/datasets/bridge5_analog_yolo/labels/test... 864 images, 78 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 864/864 1.2Kit/s 0.7s0.1ss
val: /media/hudaifa/Ventoy/datasets/bridge5_analog_yolo/images/test/dacl_dacl10k_v2_validation_0555.jpg: 1 duplicate labels removed
val: New cache created: /media/hudaifa/Ventoy/datasets/bridge5_analog_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 12.2it/s 4.4s0.2s
                   all        864       4563      0.312        0.2      0.151     0.0708
Speed: 0.2ms preprocess, 4.1ms inference, 0.0ms loss, 0.3ms postprocess per image
Ultralytics 8.4.174 🚀 Python-3.10.12 torc

In [5]:
# Cell 7 - Fine-tune model B on mixed clean + A11T-look photos (transfer learning, ~70 min)
FT = {**TRAIN, "data": str(DATA_AN / "data.yaml"), "epochs": 60, "patience": 20,
      "optimizer": "AdamW", "lr0": 0.0005, "warmup_epochs": 1}   # small learning rate: adjust, don't forget
YOLO(str(MODELS / "bridge5_y11s.pt")).train(name="bridge5_y11s_analog", batch=16, **FT)

W = RUNS / "bridge5_y11s_analog/weights/best.pt"
evaluate(W, "E4_y11s_analog")           # clean test photos      (before: 0.323)
evaluate_A11T(W, "E4_y11s_analog")      # A11T-look test photos  (before: 0.217)
shutil.copy(W, MODELS / "bridge5_y11s_analog.pt")
show()

New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7796MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/media/hudaifa/Ventoy/datasets/bridge5_analog_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixu

In [3]:
# Cell 8 - Build "drone-view" training photos: each TRAIN photo is pasted onto concrete like the
#          sim defect patches (small in the frame, slight angle), then the A11T analog look.
#          Only train/val photos are used - the sim defects come from the TEST split (no leak).
import sys, random
import cv2, numpy as np, yaml
from tqdm.auto import tqdm
sys.path.insert(0, str(Path.home() / "drone_ws/scripts"))
from analog_cam import AnalogFX

DATA_DV  = HDD / "bridge5_droneview_yolo"
TEXTURES = Path.home() / "drone_ws/sim/assets/textures"
WALLS    = [cv2.imread(str(TEXTURES / t / "albedo.jpg")) for t in ("concrete_bridge", "concrete_weathered")]
WALLS    = [w for w in WALLS if w is not None]
assert WALLS, f"no concrete textures found in {TEXTURES}"
W, H     = 640, 480                                   # drone camera image
LOOKS    = {"good": 0.3, "typical": 0.5, "bad": 0.2}
FX       = {name: AnalogFX(name, seed=i) for i, name in enumerate(LOOKS)}
P_ANALOG, P_EMPTY = 0.8, 0.08                         # 80 % analog look, 8 % plain wall (no defect)
rng = np.random.default_rng(7); random.seed(7)

def wall():
    t = random.choice(WALLS)
    s = rng.uniform(0.35, 1.0)                        # how much of the texture is in view
    ch, cw = int(t.shape[0] * s * 0.75), int(t.shape[1] * s)
    y, x = rng.integers(0, t.shape[0] - ch + 1), rng.integers(0, t.shape[1] - cw + 1)
    return cv2.resize(t[y:y + ch, x:x + cw], (W, H), interpolation=cv2.INTER_AREA).astype(np.float32)

def crop_43(w, h):                                    # same centre crop as the sim patches
    cw, ch = (round(h * 4 / 3), h) if w / h > 4 / 3 else (w, round(w * 3 / 4))
    return (w - cw) // 2, (h - ch) // 2, cw, ch

def read_boxes(p):
    return [list(map(float, l.split())) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []

def drone_view(img, boxes):
    canvas, out = wall(), []
    if img is None:
        return canvas, out
    h, w = img.shape[:2]
    x0, y0, cw, ch = crop_43(w, h)
    photo = cv2.resize(img[y0:y0 + ch, x0:x0 + cw], (640, 480), interpolation=cv2.INTER_AREA)
    pw = rng.uniform(0.18, 0.55) * W                  # patch width in the frame (sim: ~15-45 %)
    ph = pw * 0.75
    px, py = rng.uniform(0, W - pw), rng.uniform(0, H - ph)
    j = 0.08 * pw                                     # corner jitter = seen at a slight angle
    dst = np.float32([[px, py], [px + pw, py], [px + pw, py + ph], [px, py + ph]]) + rng.uniform(-j, j, (4, 2))
    dst = dst.astype(np.float32)
    src = np.float32([[0, 0], [640, 0], [640, 480], [0, 480]])
    M = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(photo, M, (W, H), flags=cv2.INTER_AREA).astype(np.float32)
    mask = cv2.warpPerspective(np.ones((480, 640), np.float32), M, (W, H))[..., None]
    canvas = canvas * (1 - mask) + warped * mask
    for c, bx, by, bw, bh in boxes:                   # labels: photo -> crop -> frame
        X0 = np.clip((bx - bw / 2) * w - x0, 0, cw); X1 = np.clip((bx + bw / 2) * w - x0, 0, cw)
        Y0 = np.clip((by - bh / 2) * h - y0, 0, ch); Y1 = np.clip((by + bh / 2) * h - y0, 0, ch)
        if (X1 - X0) * (Y1 - Y0) < 0.3 * bw * w * bh * h:   # mostly cut away by the crop
            continue
        pts = np.float32([[X0, Y0], [X1, Y0], [X1, Y1], [X0, Y1]]) * np.float32([640 / cw, 480 / ch])
        q = cv2.perspectiveTransform(pts[None], M)[0]
        qx0, qy0 = np.clip(q.min(0), 0, [W, H]); qx1, qy1 = np.clip(q.max(0), 0, [W, H])
        if qx1 - qx0 >= 4 and qy1 - qy0 >= 4:
            out.append(f"{int(c)} {(qx0 + qx1) / 2 / W:.6f} {(qy0 + qy1) / 2 / H:.6f} "
                       f"{(qx1 - qx0) / W:.6f} {(qy1 - qy0) / H:.6f}")
    return canvas, out

lists = {}
for split in ("train", "val"):
    (DATA_DV / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATA_DV / "labels" / split).mkdir(parents=True, exist_ok=True)
    made = []
    for src in tqdm(sorted((DATA5 / "images" / split).glob("*.jpg")), desc=split):
        dst = DATA_DV / "images" / split / f"dv_{src.name}"
        empty = rng.random() < P_EMPTY
        look = random.choices(list(LOOKS), list(LOOKS.values()))[0] if rng.random() < P_ANALOG else None
        if not dst.exists():
            img = None if empty else cv2.imread(str(src))
            canvas, labels = drone_view(img, read_boxes(DATA5 / "labels" / split / f"{src.stem}.txt"))
            canvas = canvas * rng.uniform(0.65, 1.15)                     # light / shadow
            frame = np.clip(canvas, 0, 255).astype(np.uint8)
            if look:
                frame = FX[look](frame)
            cv2.imwrite(str(dst), frame, [cv2.IMWRITE_JPEG_QUALITY, 92])
            (DATA_DV / "labels" / split / f"dv_{src.stem}.txt").write_text("\n".join(labels))
        made.append(str(dst))
    originals = [str(p) for p in sorted((DATA5 / "images" / split).glob("*.jpg"))]
    lists[split] = DATA_DV / f"{split}.txt"
    lists[split].write_text("\n".join(made + originals))             # drone-view + original clean photos
    print(f"{split}: {len(made)} drone-view + {len(originals)} original photos")

names = yaml.safe_load((DATA5 / "data.yaml").read_text())["names"]
(DATA_DV / "data.yaml").write_text(yaml.safe_dump(
    {"path": str(DATA_DV), "train": str(lists["train"]), "val": str(lists["val"]), "names": names}))

ex = [p for p in sorted((DATA_DV / "images/train").glob("dv_*.jpg"))      # 8 examples with their boxes
      if (DATA_DV / "labels/train" / (p.stem + ".txt")).read_text().strip()][:8]
tiles = []
for p in ex:
    im = cv2.imread(str(p))
    for l in (DATA_DV / "labels/train" / (p.stem + ".txt")).read_text().splitlines():
        c, x, y, w_, h_ = map(float, l.split())
        cv2.rectangle(im, (int((x - w_ / 2) * W), int((y - h_ / 2) * H)), (int((x + w_ / 2) * W), int((y + h_ / 2) * H)),
                      (0, 255, 0), 2)
        cv2.putText(im, names[int(c)], (int((x - w_ / 2) * W), max(14, int((y - h_ / 2) * H) - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    tiles.append(cv2.resize(im, (320, 240)))
tiles += [np.zeros_like(tiles[0])] * (8 - len(tiles))
cv2.imwrite(str(DATA_DV / "examples.jpg"), cv2.vconcat([cv2.hconcat(tiles[:4]), cv2.hconcat(tiles[4:])]))
print("dataset ready:", DATA_DV, "\nexamples:", DATA_DV / "examples.jpg")

/home/hudaifa/drone_ws/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
train: 100%|███████████████████████████████| 6408/6408 [04:30<00:00, 23.73it/s]


train: 6408 drone-view + 6408 original photos


val: 100%|███████████████████████████████████| 793/793 [00:22<00:00, 34.85it/s]


val: 793 drone-view + 793 original photos
dataset ready: /media/hudaifa/Ventoy/datasets/bridge5_droneview_yolo 
examples: /media/hudaifa/Ventoy/datasets/bridge5_droneview_yolo/examples.jpg


In [4]:
# Cell 9 - Fine-tune model B on original + drone-view photos (E5)
import shutil
FT_DV = {**TRAIN, "data": str(DATA_DV / "data.yaml"), "epochs": 40, "patience": 15, "cache": False,
         "optimizer": "AdamW", "lr0": 0.0005, "warmup_epochs": 1}
YOLO(str(MODELS / "bridge5_y11s.pt")).train(name="bridge5_y11s_droneview", batch=16, **FT_DV)
W_DV = RUNS / "bridge5_y11s_droneview/weights/best.pt"
shutil.copy(W_DV, MODELS / "bridge5_y11s_droneview.pt")
evaluate(W_DV, "E5_y11s_droneview")       # photo test, for the record
show()

New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7796MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/media/hudaifa/Ventoy/datasets/bridge5_droneview_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300,

In [3]:
# Cell 10 - E6 dataset: train photos as the drone sees them through the REAL A11T lens
#           (pasted on concrete like the sim patches -> fisheye bend + dark corners -> analog look)
#           Only train/val photos are used - the sim defects come from the TEST split (no leak).
import sys, math, random
import cv2, numpy as np, yaml
from tqdm.auto import tqdm
sys.path.insert(0, str(Path.home() / "drone_ws/scripts"))
from analog_cam import AnalogFX
from fisheye_cam import Lens

DATA_E6  = HDD / "bridge5_e6_yolo"
DATA_AN  = HDD / "bridge5_analog_yolo"                     # E4's photos (clean + analog) - kept so it doesn't forget
TEXTURES = Path.home() / "drone_ws/sim/assets/textures"
WALLS    = [cv2.imread(str(TEXTURES / t / "albedo.jpg")) for t in ("concrete_bridge", "concrete_weathered")]
WALLS    = [w for w in WALLS if w is not None]
assert WALLS, f"no concrete textures found in {TEXTURES}"
RW, RH, RHFOV = 960, 720, math.radians(136)                # same render as the simulator (camprofile real)
FR   = (RW / 2) / math.tan(RHFOV / 2)
LENS = Lens(Path.home() / "drone_ws/sim/a11t_calib.json", RHFOV)
W, H = 640, 480                                            # final drone image
LOOKS = {"good": 0.3, "typical": 0.5, "bad": 0.2}
FX = {name: AnalogFX(name, seed=i) for i, name in enumerate(LOOKS)}
P_ANALOG, P_EMPTY = 0.85, 0.08                             # 85 % analog look, 8 % plain wall (no defect)
rng = np.random.default_rng(11); random.seed(11)

def wall():
    t = random.choice(WALLS)
    s = rng.uniform(0.35, 1.0)
    ch, cw = int(t.shape[0] * s * 0.75), int(t.shape[1] * s)
    y, x = rng.integers(0, t.shape[0] - ch + 1), rng.integers(0, t.shape[1] - cw + 1)
    return cv2.resize(t[y:y + ch, x:x + cw], (RW, RH), interpolation=cv2.INTER_AREA).astype(np.float32)

def crop_43(w, h):                                         # same centre crop as the sim patches
    cw, ch = (round(h * 4 / 3), h) if w / h > 4 / 3 else (w, round(w * 3 / 4))
    return (w - cw) // 2, (h - ch) // 2, cw, ch

def read_boxes(p):
    return [list(map(float, l.split())) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []

def to_lens(pts):                                          # render pixels -> real-lens image pixels
    return np.array([LENS.norm_to_pix((u - RW / 2) / FR, (v - RH / 2) / FR) for u, v in pts])

def drone_view(img, boxes):
    canvas, out = wall(), []
    if img is None:
        return canvas, out
    h, w = img.shape[:2]
    x0, y0, cw, ch = crop_43(w, h)
    photo = cv2.resize(img[y0:y0 + ch, x0:x0 + cw], (640, 480), interpolation=cv2.INTER_AREA)
    pw = rng.uniform(0.18, 0.55) * W                       # patch width (render centre ~ same scale as the lens image)
    ph = pw * 0.75
    px = rng.uniform(RW / 2 - 360, RW / 2 + 360 - pw)      # anywhere the lens can see, edges included
    py = rng.uniform(RH / 2 - 260, RH / 2 + 260 - ph)
    j = 0.08 * pw                                          # seen at a slight angle
    dst = (np.float32([[px, py], [px + pw, py], [px + pw, py + ph], [px, py + ph]])
           + rng.uniform(-j, j, (4, 2))).astype(np.float32)
    M = cv2.getPerspectiveTransform(np.float32([[0, 0], [640, 0], [640, 480], [0, 480]]), dst)
    warped = cv2.warpPerspective(photo, M, (RW, RH), flags=cv2.INTER_AREA).astype(np.float32)
    mask = cv2.warpPerspective(np.ones((480, 640), np.float32), M, (RW, RH))[..., None]
    canvas = canvas * (1 - mask) + warped * mask
    t = np.linspace(0, 1, 9)
    for c, bx, by, bw, bh in boxes:                        # labels: photo -> crop -> render -> lens image
        X0 = np.clip((bx - bw / 2) * w - x0, 0, cw); X1 = np.clip((bx + bw / 2) * w - x0, 0, cw)
        Y0 = np.clip((by - bh / 2) * h - y0, 0, ch); Y1 = np.clip((by + bh / 2) * h - y0, 0, ch)
        if (X1 - X0) * (Y1 - Y0) < 0.3 * bw * w * bh * h:  # mostly cut away by the crop
            continue
        xs = np.r_[X0 + (X1 - X0) * t, np.full(9, X1), X1 - (X1 - X0) * t, np.full(9, X0)]
        ys = np.r_[np.full(9, Y0), Y0 + (Y1 - Y0) * t, np.full(9, Y1), Y1 - (Y1 - Y0) * t]
        pts = np.stack([xs * 640 / cw, ys * 480 / ch], 1).astype(np.float32)
        q = to_lens(cv2.perspectiveTransform(pts[None], M)[0])     # the box edge bends -> take its bounding box
        full = (q[:, 0].max() - q[:, 0].min()) * (q[:, 1].max() - q[:, 1].min())
        qx0, qy0 = np.clip(q.min(0), 0, [W, H]); qx1, qy1 = np.clip(q.max(0), 0, [W, H])
        if qx1 - qx0 >= 4 and qy1 - qy0 >= 4 and (qx1 - qx0) * (qy1 - qy0) >= 0.3 * full:
            out.append(f"{int(c)} {(qx0 + qx1) / 2 / W:.6f} {(qy0 + qy1) / 2 / H:.6f} "
                       f"{(qx1 - qx0) / W:.6f} {(qy1 - qy0) / H:.6f}")
    return canvas, out

lists = {}
for split in ("train", "val"):
    (DATA_E6 / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATA_E6 / "labels" / split).mkdir(parents=True, exist_ok=True)
    made = []
    for src in tqdm(sorted((DATA5 / "images" / split).glob("*.jpg")), desc=split):
        dst = DATA_E6 / "images" / split / f"e6_{src.name}"
        empty = rng.random() < P_EMPTY
        look = random.choices(list(LOOKS), list(LOOKS.values()))[0] if rng.random() < P_ANALOG else None
        if not dst.exists():
            img = None if empty else cv2.imread(str(src))
            canvas, labels = drone_view(img, read_boxes(DATA5 / "labels" / split / f"{src.stem}.txt"))
            frame = np.clip(canvas * rng.uniform(0.65, 1.15), 0, 255).astype(np.uint8)   # light / shadow
            frame = LENS.apply(frame)                      # real fisheye + dark corners -> 640x480
            if look:
                frame = FX[look](frame)
            cv2.imwrite(str(dst), frame, [cv2.IMWRITE_JPEG_QUALITY, 92])
            (DATA_E6 / "labels" / split / f"e6_{src.stem}.txt").write_text("\n".join(labels))
        made.append(str(dst))
    e4 = [str(p) for p in sorted((DATA_AN / "images" / split).glob("*.jpg"))]
    lists[split] = DATA_E6 / f"{split}.txt"
    lists[split].write_text("\n".join(made + e4))
    print(f"{split}: {len(made)} real-lens drone views + {len(e4)} E4 photos")

names = yaml.safe_load((DATA5 / "data.yaml").read_text())["names"]
(DATA_E6 / "data.yaml").write_text(yaml.safe_dump(
    {"path": str(DATA_E6), "train": str(lists["train"]), "val": str(lists["val"]), "names": names}))

ex = [p for p in sorted((DATA_E6 / "images/train").glob("e6_*.jpg"))
      if (DATA_E6 / "labels/train" / (p.stem + ".txt")).read_text().strip()][:8]
tiles = []
for p in ex:
    im = cv2.imread(str(p))
    for l in (DATA_E6 / "labels/train" / (p.stem + ".txt")).read_text().splitlines():
        c, x, y, w_, h_ = map(float, l.split())
        cv2.rectangle(im, (int((x - w_ / 2) * W), int((y - h_ / 2) * H)), (int((x + w_ / 2) * W), int((y + h_ / 2) * H)),
                      (0, 255, 0), 2)
        cv2.putText(im, names[int(c)], (int((x - w_ / 2) * W), max(14, int((y - h_ / 2) * H) - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    tiles.append(cv2.resize(im, (320, 240)))
tiles += [np.zeros((240, 320, 3), np.uint8)] * (8 - len(tiles))
cv2.imwrite(str(DATA_E6 / "examples.jpg"), cv2.vconcat([cv2.hconcat(tiles[:4]), cv2.hconcat(tiles[4:])]))
print("dataset ready:", DATA_E6, "\nexamples:", DATA_E6 / "examples.jpg")

/home/hudaifa/drone_ws/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
train: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6408/6408 [05:19<00:00, 20.07it/s]


train: 6408 real-lens drone views + 6408 E4 photos


val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 793/793 [00:30<00:00, 25.63it/s]


val: 793 real-lens drone views + 793 E4 photos
dataset ready: /media/hudaifa/Ventoy/datasets/bridge5_e6_yolo 
examples: /media/hudaifa/Ventoy/datasets/bridge5_e6_yolo/examples.jpg


In [4]:
# Cell 11 - Train E6: start from E4 (already knows analog) on real-lens drone views + E4 photos
import shutil
FT_E6 = {**TRAIN, "data": str(DATA_E6 / "data.yaml"), "epochs": 40, "patience": 15, "cache": False,
         "optimizer": "AdamW", "lr0": 0.0005, "warmup_epochs": 1}
YOLO(str(MODELS / "bridge5_y11s_analog.pt")).train(name="bridge5_y11s_e6", batch=16, **FT_E6)
W_E6 = RUNS / "bridge5_y11s_e6/weights/best.pt"
shutil.copy(W_E6, MODELS / "bridge5_y11s_e6.pt")
evaluate(W_E6, "E6_y11s_reallens")       # photo test, for the record
show()

New https://pypi.org/project/ultralytics/8.4.215 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7796MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/media/hudaifa/Ventoy/datasets/bridge5_e6_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=